# ArXiv RAG System — Pipeline Development

**End-to-end walkthrough:** data ingestion → semantic chunking → embedding → vector store indexing → retrieval → generation.

This notebook is the single executable record of every step taken to build the system. Run cells top-to-bottom on a Kaggle GPU instance (T4 or P100) to reproduce the full index.

---

| Step | Module | Output |
|------|--------|--------|
| 1 | `data_processing.py` | `arxiv_clean.parquet` — 10,000 clean abstracts |
| 2 | `embedding_pipeline.py` | ChromaDB — 10K+ chunks, 384-dim vectors |
| 3 | `retrieval.py` | Verified semantic search + MMR reranking |
| 4 | `generation.py` | Full RAG answer with Author-Year citations |


## 0. Environment Check

In [ ]:
import sys, os, subprocess
from pathlib import Path

# ── GPU check ────────────────────────────────────────────────────────
try:
    import torch
    gpu_available = torch.cuda.is_available()
    gpu_name      = torch.cuda.get_device_name(0) if gpu_available else 'None'
except ImportError:
    gpu_available, gpu_name = False, 'torch not installed'

print(f'Python  : {sys.version.split()[0]}')
print(f'GPU     : {gpu_name}')
print(f'CUDA    : {gpu_available}')
print(f'CWD     : {Path.cwd()}')

# ── Working directory ─────────────────────────────────────────────────
BASE = Path('/kaggle/working')
os.chdir(BASE)
sys.path.insert(0, str(BASE))
print(f'Base    : {BASE}')

# ── Required dirs ─────────────────────────────────────────────────────
for d in ['data/processed', 'logs', 'chroma_db']:
    (BASE / d).mkdir(parents=True, exist_ok=True)
print('Dirs    : OK')

In [ ]:
# Verify all critical imports resolve — install missing packages if needed
required = [
    'sentence_transformers', 'chromadb', 'transformers',
    'fastapi', 'pydantic', 'yaml', 'nltk', 'tqdm',
]
missing = []
for pkg in required:
    try:
        __import__(pkg)
        print(f'  ✅  {pkg}')
    except ImportError:
        print(f'  ❌  {pkg}  ← missing')
        missing.append(pkg)

if missing:
    print(f'\nInstalling {missing}...')
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                    *missing], check=True)
else:
    print('\nAll dependencies satisfied ✅')

In [ ]:
import yaml

config = yaml.safe_load((BASE / 'config.yaml').read_text())

# Surface key settings for the notebook run
print('Configuration:')
print(f"  embedding model : {config['embedding']['model_name']}")
print(f"  dimensions      : {config['embedding']['dimensions']}")
print(f"  vector store    : {config['vector_store']['type']}")
print(f"  indexing target : {config['indexing']['total_target']:,} documents")
print(f"  date range      : {config['indexing']['date_start']} → {config['indexing']['date_end']}")
print(f"  categories      : {list(config['indexing']['categories'].keys())}")
print(f"  llm mode        : {config['llm']['mode']}")
print(f"  llm primary     : {config['llm']['primary']}")
print(f"  llm fallback    : {config['llm']['fallback']}")

## 1. Data Processing

**Module:** `src/data_processing.py`  
**Input:** Raw arXiv JSONL (2.4M+ papers)  
**Output:** `data/processed/arxiv_clean.parquet` — 10,000 clean abstracts

### Why streaming?
The full arXiv snapshot is ~3.5GB. Loading it into RAM at once would OOM on Kaggle's 13GB limit. `load_arxiv_jsonl()` yields 10,000-row DataFrames so peak RAM stays under 1GB during filtering.

### Why stratified sampling?
Uniform random sampling over-represents large categories (cs.LG has 10x more papers than cs.RO). Stratification ensures every domain contributes meaningfully to retrieval diversity.

In [ ]:
import pandas as pd
from src.data_processing import (
    load_arxiv_jsonl,
    filter_categories,
    filter_date_range,
    validate_records,
    stratified_sample,
    build_metadata_columns,
    export_parquet,
)

RAW_PATH       = config['indexing'].get(
    'raw_path',
    '/kaggle/input/arxiv/arxiv-metadata-oag-snapshot.json'
)
PROCESSED_PATH = BASE / 'data/processed/arxiv_clean.parquet'
CATEGORIES     = list(config['indexing']['categories'].keys())
DATE_START     = config['indexing']['date_start']
DATE_END       = config['indexing']['date_end']
TOTAL_TARGET   = config['indexing']['total_target']
BUFFER_FACTOR  = config['indexing']['buffer_factor']
CAT_WEIGHTS    = config['indexing']['categories']

print(f'Raw data path   : {RAW_PATH}')
print(f'Processed path  : {PROCESSED_PATH}')
print(f'Target records  : {TOTAL_TARGET:,}')

In [ ]:
# ── Stream, filter, validate ───────────────────────────────────────────
if PROCESSED_PATH.exists():
    print(f'Processed file already exists — loading from cache.')
    df = pd.read_parquet(PROCESSED_PATH)
    print(f'Loaded {len(df):,} records from {PROCESSED_PATH}')
else:
    print('Processing raw arXiv data...')
    chunks = []
    for chunk in load_arxiv_jsonl(RAW_PATH):
        filtered = filter_categories(chunk, CATEGORIES)
        filtered = filter_date_range(filtered, DATE_START, DATE_END)
        if len(filtered) > 0:
            chunks.append(filtered)

    df_raw = pd.concat(chunks, ignore_index=True)
    print(f'After category + date filter : {len(df_raw):,} records')

    df_valid = validate_records(df_raw)
    print(f'After validation             : {len(df_valid):,} records')

    sample_n = int(TOTAL_TARGET * BUFFER_FACTOR)
    df_sampled = stratified_sample(df_valid, n=sample_n,
                                   category_weights=CAT_WEIGHTS)
    print(f'After stratified sample      : {len(df_sampled):,} records')

    df = build_metadata_columns(df_sampled)
    export_parquet(df, str(PROCESSED_PATH))
    print(f'Saved to {PROCESSED_PATH}')

In [ ]:
# ── Data quality summary ───────────────────────────────────────────────
print('Dataset summary')
print('─' * 50)
print(f'  Total records    : {len(df):,}')
print(f'  Columns          : {list(df.columns)}')
print(f'  Date range       : {df["year"].min()} – {df["year"].max()}')
print()
print('Category distribution:')
cat_counts = df['category'].value_counts()
for cat, count in cat_counts.items():
    pct = count / len(df) * 100
    bar = '█' * int(pct / 2)
    print(f'  {cat:<10} {count:>5,}  ({pct:5.1f}%)  {bar}')
print()
print('Abstract length stats (words):')
word_counts = df['abstract'].str.split().str.len()
print(f'  min={word_counts.min()}  '
      f'median={word_counts.median():.0f}  '
      f'mean={word_counts.mean():.0f}  '
      f'max={word_counts.max()}')
print()
print(df[['paper_id','title','citation_str','year','category']].head(3).to_string())

## 2. Indexing Pipeline

**Module:** `src/embedding_pipeline.py`, `src/vector_store.py`  
**Input:** `arxiv_clean.parquet`  
**Output:** Populated ChromaDB collection

### Semantic chunking rationale
Fixed-size chunking splits mid-sentence and mixes topics. Semantic chunking detects topic shifts via cosine similarity drops between consecutive sentences — keeping related content together improves embedding coherence and downstream retrieval precision.

### Checkpoint design
Every 1,000 documents the pipeline saves a JSON checkpoint of indexed paper IDs. On restart, already-indexed IDs are skipped. This makes the pipeline crash-safe on Kaggle's 12-hour session limit.

In [ ]:
from src.embedding_pipeline import EmbeddingModel, SemanticChunker, IndexingPipeline
from src.vector_store import ChromaVectorStore

# ── Embedding model ────────────────────────────────────────────────────
embedder = EmbeddingModel(
    model_name=config['embedding']['model_name'],
    device=config['embedding']['device'],
)
embedder.load()
assert embedder.dimensions == 384, f'Expected 384, got {embedder.dimensions}'
print(f'Embedding model : {config["embedding"]["model_name"]}')
print(f'Dimensions      : {embedder.dimensions}')
print(f'Device          : {embedder.device}')

In [ ]:
# ── Semantic chunker ───────────────────────────────────────────────────
chunker = SemanticChunker(
    model=embedder.model,
    similarity_threshold=config['chunking']['similarity_threshold'],
    min_chunk_tokens=config['chunking']['min_chunk_tokens'],
    max_chunk_tokens=config['chunking']['max_chunk_tokens'],
    overlap_sentences=config['chunking']['overlap_sentences'],
)

# Quick sanity check on one abstract
sample_abstract = df['abstract'].iloc[0]
sample_chunks   = chunker.chunk(sample_abstract)
print(f'Sample abstract ({len(sample_abstract.split())} words) → {len(sample_chunks)} chunks')
for i, ch in enumerate(sample_chunks):
    print(f'  chunk {i}: {len(ch.split())} words | {ch[:80]}...')

In [ ]:
# ── Vector store ───────────────────────────────────────────────────────
chroma_dir  = str(BASE / config['vector_store']['chroma_persist_dir'].lstrip('./'))
vector_store = ChromaVectorStore(
    persist_directory=chroma_dir,
    collection_name='arxiv_papers',
)
vector_store.connect()
print(f'Vector store    : ChromaDB')
print(f'Persist dir     : {chroma_dir}')
print(f'Existing count  : {vector_store.count():,} chunks')

In [ ]:
# ── Run indexing pipeline ──────────────────────────────────────────────
CHECKPOINT_PATH = BASE / 'data/processed/indexing_checkpoint.json'

pipeline = IndexingPipeline(
    embedder=embedder,
    chunker=chunker,
    vector_store=vector_store,
    checkpoint_path=str(CHECKPOINT_PATH),
    embed_batch_size=config['indexing']['embed_batch_size'],
    doc_batch_size=config['indexing']['doc_batch_size'],
    checkpoint_interval=config['indexing']['checkpoint_interval'],
)

report = pipeline.run(df)

print('\nIndexing complete!')
print(f'  Documents indexed : {report.total_documents:,}')
print(f'  Total chunks      : {report.total_chunks:,}')
print(f'  Avg chunks/doc    : {report.avg_chunks_per_doc:.2f}')
print(f'  Failed documents  : {report.failed_documents}')
print(f'  Total time        : {report.total_time_seconds:.1f}s')
print(f'  Embed time        : {report.embedding_time_seconds:.1f}s')
print(f'  Upsert time       : {report.upsert_time_seconds:.1f}s')

In [ ]:
# ── Verify index ───────────────────────────────────────────────────────
final_count = vector_store.count()
print(f'ChromaDB chunk count : {final_count:,}')
assert final_count >= 10_000, (
    f'Index too small: {final_count} < 10,000'
)
print('Index size assertion  : ✅ ≥ 10,000 chunks')

## 3. Retrieval Verification

**Module:** `src/retrieval.py`

Verifies that semantic search returns relevant results and that MMR reranking produces topically diverse top-5 chunks across distinct papers.

### MMR intuition
Without MMR, the top-5 results are often 4-5 chunks from the *same* paper — redundant for generation context. MMR penalises similarity to already-selected results, forcing diversity while preserving relevance (λ=0.7).

In [ ]:
from src.retrieval import DocumentRetriever

retriever = DocumentRetriever(
    vector_store=vector_store,
    embedder=embedder,
    top_k=config['retrieval']['top_k'],
    score_threshold=config['retrieval']['score_threshold'],
    mmr_lambda=config['retrieval']['mmr_lambda'],
)

TEST_QUERIES = [
    'What is the attention mechanism in transformers?',
    'How does few-shot learning work?',
    'What methods improve object detection accuracy?',
    'How are large language models trained?',
]

for query in TEST_QUERIES:
    result = retriever.retrieve(query)
    paper_ids = [d.paper_id for d in result.documents]
    unique    = len(set(paper_ids))
    scores    = [f'{d.score:.3f}' for d in result.documents]
    print(f'Query   : {query[:60]}')
    print(f'  docs  : {len(result.documents)} | unique papers: {unique}')
    print(f'  scores: {scores}')
    print(f'  low_confidence: {result.low_confidence}')
    print(f'  latency: {result.retrieval_latency_ms:.1f}ms')
    print()

In [ ]:
# ── Diversity check: no two results from same paper_id ─────────────────
print('Deduplication verification:')
all_passed = True
for query in TEST_QUERIES:
    result   = retriever.retrieve(query)
    ids      = [d.paper_id for d in result.documents]
    no_dupes = len(ids) == len(set(ids))
    status   = '✅' if no_dupes else '❌'
    print(f'  {status}  {query[:55]} | paper_ids unique: {no_dupes}')
    if not no_dupes:
        all_passed = False

print(f'\n  Deduplication: {"✅ All queries" if all_passed else "❌ Failures found"}')

## 4. Generation — Full RAG Pipeline

**Module:** `src/generation.py`

Loads the LLM backend (Mistral-7B on GPU, flan-t5-large on CPU), runs the full RAG pipeline, and verifies citation formatting.

### Citation discipline
Every factual paragraph must contain at least one Author-Year citation drawn from the retrieved sources. `CitationFormatter.validate_citations()` cross-references every `(Author et al., YYYY)` pattern in the answer against the source list — hallucinated citations are flagged.

In [ ]:
from src.generation import LLMBackend, PromptBuilder, CitationFormatter, RAGPipeline

# ── Load LLM ──────────────────────────────────────────────────────────
llm = LLMBackend(config=config)
llm.load()
print(f'LLM backend : {llm.backend_name}')
print(f'Loaded      : {llm.is_loaded}')

In [ ]:
# ── Build full RAG pipeline ────────────────────────────────────────────
prompt_builder     = PromptBuilder(config=config)
citation_formatter = CitationFormatter()

rag = RAGPipeline(
    retriever=retriever,
    llm=llm,
    prompt_builder=prompt_builder,
    citation_formatter=citation_formatter,
)

In [ ]:
# ── Single query test ─────────────────────────────────────────────────
query    = 'What is the attention mechanism and why is it important for NLP?'
response = rag.query(question=query)

print(f'Query       : {query}')
print(f'Model used  : {response.model_used}')
print(f'Confidence  : {response.confidence:.3f}')
print(f'Low conf?   : {response.low_confidence}')
print(f'Retrieval   : {response.retrieval_latency_ms:.1f}ms')
print(f'Generation  : {response.generation_latency_ms:.1f}ms')
print(f'Total       : {response.total_latency_ms:.1f}ms')
print()
print('Answer:')
print('─' * 60)
print(response.answer)
print('─' * 60)
print()
print(f'Citations   : {response.citations}')
print(f'Sources     : {len(response.sources)}')
for src in response.sources:
    print(f'  [{src.score:.3f}] {src.citation_str} — {src.metadata.get("title", "")[:60]}')

In [ ]:
# ── Multi-turn conversation test ───────────────────────────────────────
print('Multi-turn conversation test')
print('=' * 60)

history = []
turns = [
    'What is few-shot learning?',
    'How does it compare to zero-shot learning?',
    'Which model architectures support this best?',
]

for i, turn_query in enumerate(turns, 1):
    print(f'\nTurn {i}: {turn_query}')
    response = rag.query(question=turn_query, conversation_history=history)
    history.append({'role': 'user',      'content': turn_query})
    history.append({'role': 'assistant', 'content': response.answer})
    print(f'Answer  : {response.answer[:200]}...')
    print(f'Citations: {response.citations}')
    print(f'Latency : {response.total_latency_ms:.0f}ms')

print(f'\nHistory length : {len(history)} messages ({len(history)//2} turns)')

In [ ]:
# ── Citation validation ────────────────────────────────────────────────
print('Citation validation across 5 queries:')
print('─' * 60)

test_queries = [
    'What is self-supervised learning?',
    'How do convolutional neural networks work?',
    'What is reinforcement learning from human feedback?',
    'How does BERT represent language?',
    'What are diffusion models?',
]

all_valid = True
for q in test_queries:
    resp   = rag.query(question=q)
    valid  = citation_formatter.validate_citations(resp.answer, resp.sources)
    status = '✅' if valid else '❌'
    print(f'  {status}  {q[:55]}')
    print(f'       citations: {resp.citations}')
    if not valid:
        all_valid = False

print(f'\nCitation accuracy: {"✅ All valid" if all_valid else "❌ Hallucinations detected"}')

## 5. API Smoke Test

Starts the FastAPI server in the background and runs requests against it to verify the full stack end-to-end: HTTP → FastAPI → RAG → response.

In [ ]:
import time, requests, threading, uvicorn
from api.main import app

# ── Start FastAPI in background thread ────────────────────────────────
server = uvicorn.Server(uvicorn.Config(
    app=app, host='127.0.0.1', port=8000, log_level='error'
))
thread = threading.Thread(target=server.run, daemon=True)
thread.start()
time.sleep(3)  # allow startup

BASE_URL = 'http://127.0.0.1:8000'

# ── Health check ──────────────────────────────────────────────────────
health = requests.get(f'{BASE_URL}/health').json()
print('Health check:')
for k, v in health.items():
    print(f'  {k:<20}: {v}')

In [ ]:
# ── Query endpoint ────────────────────────────────────────────────────
payload  = {'query': 'What is the attention mechanism?', 'top_k': 5}
resp     = requests.post(f'{BASE_URL}/query', json=payload)
data     = resp.json()

print(f'Status code     : {resp.status_code}')
print(f'Answer excerpt  : {data["answer"][:150]}...')
print(f'Citations       : {data["citations"]}')
print(f'Latency         : {data["latency_ms"]:.1f}ms')
print(f'Session ID      : {data["session_id"]}')
print(f'Turn number     : {data["turn_number"]}')
print(f'Low confidence  : {data["low_confidence"]}')
assert resp.status_code == 200
assert 'answer' in data
assert 'citations' in data
print('\nAPI smoke test: ✅ PASSED')

## 6. Pipeline Summary

| Component | Status | Key Metric |
|-----------|--------|------------|
| Data processing | ✅ | 10,000 clean abstracts |
| Semantic chunking | ✅ | ~1.5 chunks/abstract avg |
| Embedding model | ✅ | 384-dim, all-MiniLM-L6-v2 |
| ChromaDB index | ✅ | ≥ 10,000 chunks indexed |
| MMR retrieval | ✅ | top-5 diverse, deduplicated |
| LLM generation | ✅ | Author-Year cited answers |
| Multi-turn | ✅ | 6-turn memory window |
| FastAPI | ✅ | /query /health /history /clear |

**Next:** Run `02_evaluation.ipynb` to measure retrieval and generation, |
quality against the locked success metrics.